## Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("✓ Imports complete")

✓ Imports complete


## Configuration

In [3]:
# Input paths
MDRAC_INPUT = 'results/mdrac_conflicts.csv'
SPF_INPUT = 'results/spf_conflicts.csv'

# Output paths
MDRAC_OUTPUT = 'results/postprocessed_conflicts/mdrac_conflicts_processed.csv'
SPF_OUTPUT = 'results/postprocessed_conflicts/spf_conflicts_processed.csv'

# Filter thresholds
MAX_YAW_DIFF = 160.0  # degrees - exclude near head-on conflicts

print(f"Configuration:")
print(f"  Max yaw difference: {MAX_YAW_DIFF}°")
print(f"  M-DRAC input: {MDRAC_INPUT}")
print(f"  SPF input: {SPF_INPUT}")

Configuration:
  Max yaw difference: 160.0°
  M-DRAC input: results/mdrac_conflicts.csv
  SPF input: results/spf_conflicts.csv


---
## Utility Functions

In [4]:
def create_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create normalized pair_id (always smaller ID first).
    Ensures (id1=100, id2=200) and (id1=200, id2=100) are treated as same pair.
    """
    df = df.copy()
    
    id_min = np.minimum(df['id1'].values, df['id2'].values)
    id_max = np.maximum(df['id1'].values, df['id2'].values)
    
    df['pair_id'] = id_min.astype(str) + '_' + id_max.astype(str)
    
    return df


def apply_yaw_filter(df: pd.DataFrame, max_yaw_diff: float) -> pd.DataFrame:
    """
    Filter conflicts by yaw difference threshold.
    Removes near head-on conflicts (opposite directions).
    """
    initial_count = len(df)
    filtered = df[df['yaw_diff'] < max_yaw_diff].copy()
    removed = initial_count - len(filtered)
    
    print(f"  Yaw filter (< {max_yaw_diff}°): {removed:,} removed ({removed/initial_count*100:.1f}%)")
    return filtered


def deduplicate_pairs(df: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    """
    Group by pair_id and select worst case.
    
    Args:
        df: DataFrame with pair_id column
        metric_col: Column to maximize ('MDRAC' or 'composite_risk')
    
    Returns:
        Deduplicated DataFrame (one row per pair)
    """
    initial_count = len(df)
    unique_pairs = df['pair_id'].nunique()
    
    # Sort by metric (desc) and distance (asc) for tiebreaker
    df_sorted = df.sort_values([metric_col, 'dist'], ascending=[False, True])
    
    # Keep first row per pair (worst case)
    deduplicated = df_sorted.groupby('pair_id', as_index=False).first()
    
    reduction = (1 - len(deduplicated) / initial_count) * 100
    print(f"  Deduplication: {initial_count:,} → {len(deduplicated):,} ({reduction:.1f}% reduction)")
    print(f"  Avg detections per pair: {initial_count/unique_pairs:.1f}")
    
    return deduplicated


---
# M-DRAC Post-Processing

## Load M-DRAC Detections

In [5]:
print("="*70)
print("M-DRAC POST-PROCESSING")
print("="*70)

mdrac_raw = pd.read_csv(MDRAC_INPUT)
print(f"\nLoaded {len(mdrac_raw):,} raw M-DRAC detections")
print(f"Columns: {list(mdrac_raw.columns)}")
print(f"\nSample:")
mdrac_raw.head(3)

M-DRAC POST-PROCESSING

Loaded 103 raw M-DRAC detections
Columns: ['timestamp', 'id1', 'id2', 'interaction', 'leader', 'dist', 'TTC', 'MDRAC', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,interaction,leader,dist,TTC,MDRAC,closing_speed,speed_diff,yaw_diff,link
0,2025-06-01 00:06:43.443032980,10287539,10287921,bus_v_car,10287921,7.745415,1.154008,inf,6.711750,1.796619,175.88795,https://di-india-collab.flow-analytics.io/tool...
1,2025-06-01 00:06:43.537080050,10287539,10287921,bus_v_car,10287921,7.170718,1.100740,inf,6.514453,1.958039,179.25912,https://di-india-collab.flow-analytics.io/tool...
2,2025-06-01 01:50:10.549896002,10326165,10326177,car_v_car,10326177,5.484753,0.578078,inf,9.487919,6.011283,167.29920,https://di-india-collab.flow-analytics.io/tool...


## Apply Filters

In [6]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
mdrac_filtered = apply_yaw_filter(mdrac_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
mdrac_filtered = create_pair_id(mdrac_filtered)

# Step 3: Deduplicate by pair (keep worst MDRAC)
mdrac_processed = deduplicate_pairs(mdrac_filtered, metric_col='MDRAC')

# Remove temporary pair_id column
mdrac_output = mdrac_processed.drop(columns=['pair_id'])

print(f"\n✓ M-DRAC processing complete")
print(f"  Final conflicts: {len(mdrac_output):,}")
print(f"  Total reduction: {(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 160.0°): 70 removed (68.0%)
  Deduplication: 33 → 11 (66.7% reduction)
  Avg detections per pair: 3.0

✓ M-DRAC processing complete
  Final conflicts: 11
  Total reduction: 89.3%


## M-DRAC Analysis

In [7]:
print("\n" + "="*70)
print("M-DRAC ANALYSIS")
print("="*70)

print("\nMDRAC Statistics:")
print(mdrac_output['MDRAC'].describe())

print("\nInteraction Type Distribution:")
print(mdrac_output['interaction'].value_counts())

print("\nTop 10 Highest MDRAC Conflicts:")
top10_mdrac = mdrac_output.nlargest(10, 'MDRAC')[[
    'timestamp', 'id1', 'id2', 'interaction', 'leader', 
    'dist', 'TTC', 'MDRAC', 'yaw_diff'
]]
print(top10_mdrac.to_string(index=False))


M-DRAC ANALYSIS

MDRAC Statistics:
count    11.000000
mean           inf
std            NaN
min       9.416742
25%      75.397912
50%            NaN
75%            NaN
max            inf
Name: MDRAC, dtype: float64

Interaction Type Distribution:
interaction
car_v_car      5
bus_v_van      1
truck_v_car    1
truck_v_van    1
bus_v_car      1
car_v_van      1
car_v_truck    1
Name: count, dtype: int64

Top 10 Highest MDRAC Conflicts:
                    timestamp      id1      id2 interaction   leader     dist      TTC     MDRAC   yaw_diff
2025-06-01 07:48:46.214777946 10471095 10471128   bus_v_van 10471095 3.536639 1.125174       inf   0.671435
2025-06-01 09:01:54.258215904 10517244 10517431 truck_v_car 10517244 4.926697 0.655279       inf   0.000000
2025-06-01 09:04:00.123322010 10518668 10518739   car_v_car 10518739 4.958520 0.425021       inf  85.607950
2025-06-01 09:35:30.561595917 10540034 10540130   car_v_car 10540130 4.248736 0.341212       inf 159.046140
2025-06-01 12:34:40.11

/home/ubuntu/prem/.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/ubuntu/prem/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


## Save M-DRAC Results

In [8]:
mdrac_output.to_csv(MDRAC_OUTPUT, index=False)
print(f"\n✓ Saved M-DRAC processed conflicts to: {MDRAC_OUTPUT}")
print(f"  Rows: {len(mdrac_output):,}")


✓ Saved M-DRAC processed conflicts to: results/postprocessed_conflicts/mdrac_conflicts_processed.csv
  Rows: 11


---
# SPF Post-Processing

## Load SPF Detections

In [9]:
print("\n" + "="*70)
print("SPF POST-PROCESSING")
print("="*70)

spf_raw = pd.read_csv(SPF_INPUT)
print(f"\nLoaded {len(spf_raw):,} raw SPF detections")
print(f"Columns: {list(spf_raw.columns)}")
print(f"\nSample:")
spf_raw.head(3)


SPF POST-PROCESSING

Loaded 2,349 raw SPF detections
Columns: ['timestamp', 'id1', 'id2', 'interaction', 'dist', 'TTC', 'composite_risk', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,interaction,dist,TTC,composite_risk,closing_speed,speed_diff,yaw_diff,link
0,2025-06-01 00:07:50.832639933,10287921,10288098,car_v_car,3.781766,0.914998,0.969074,4.133087,3.848074,2.461928,https://di-india-collab.flow-analytics.io/tool...
1,2025-06-01 00:07:50.941036940,10287921,10288098,car_v_car,3.313090,0.762758,0.989730,4.343566,3.944590,5.371479,https://di-india-collab.flow-analytics.io/tool...
2,2025-06-01 00:07:51.038043976,10287921,10288098,car_v_car,2.912292,0.642655,0.993468,4.531657,4.047575,6.266725,https://di-india-collab.flow-analytics.io/tool...


## Apply Filters

In [10]:
print(f"\nApplying filters...")

# Step 1: Yaw filter
spf_filtered = apply_yaw_filter(spf_raw, MAX_YAW_DIFF)

# Step 2: Create pair IDs
spf_filtered = create_pair_id(spf_filtered)

# Step 3: Deduplicate by pair (keep highest composite_risk)
spf_processed = deduplicate_pairs(spf_filtered, metric_col='composite_risk')

# Remove temporary pair_id column
spf_output = spf_processed.drop(columns=['pair_id'])

print(f"\n✓ SPF processing complete")
print(f"  Final conflicts: {len(spf_output):,}")
print(f"  Total reduction: {(1 - len(spf_output)/len(spf_raw))*100:.1f}%")


Applying filters...
  Yaw filter (< 160.0°): 43 removed (1.8%)
  Deduplication: 2,306 → 349 (84.9% reduction)
  Avg detections per pair: 6.6

✓ SPF processing complete
  Final conflicts: 349
  Total reduction: 85.1%


## SPF Analysis

In [11]:
print("\n" + "="*70)
print("SPF ANALYSIS")
print("="*70)

print("\nComposite Risk Statistics:")
print(spf_output['composite_risk'].describe())

print("\nInteraction Type Distribution:")
print(spf_output['interaction'].value_counts())

# print("\nTop 10 Highest Risk Conflicts:")
# top10_spf = spf_output.nlargest(10, 'composite_risk')[[
#     'timestamp', 'id1', 'id2', 'interaction', 
#     'dist', 'TTC', 'composite_risk', 'yaw_diff'
# ]]
# print(top10_spf.to_string(index=False))


SPF ANALYSIS

Composite Risk Statistics:
count    349.000000
mean       0.712086
std        0.144421
min        0.500032
25%        0.595592
50%        0.682678
75%        0.813575
max        1.000000
Name: composite_risk, dtype: float64

Interaction Type Distribution:
interaction
car_v_car      302
van_v_car       20
car_v_van       15
bus_v_car        3
truck_v_car      2
bus_v_van        2
van_v_van        1
bus_v_truck      1
truck_v_van      1
van_v_truck      1
car_v_truck      1
Name: count, dtype: int64


## Save SPF Results

In [12]:
spf_output.to_csv(SPF_OUTPUT, index=False)
print(f"\n✓ Saved SPF processed conflicts to: {SPF_OUTPUT}")
print(f"  Rows: {len(spf_output):,}")


✓ Saved SPF processed conflicts to: results/postprocessed_conflicts/spf_conflicts_processed.csv
  Rows: 349


---
# Summary Statistics

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

summary = pd.DataFrame({
    'Method': ['M-DRAC', 'SPF'],
    'Raw Detections': [len(mdrac_raw), len(spf_raw)],
    'After Yaw Filter': [len(mdrac_filtered), len(spf_filtered)],
    'After Deduplication': [len(mdrac_output), len(spf_output)],
    'Total Reduction %': [
        f"{(1 - len(mdrac_output)/len(mdrac_raw))*100:.1f}%",
        f"{(1 - len(spf_output)/len(spf_raw))*100:.1f}%"
    ]
})

print("\n", summary.to_string(index=False))


FINAL SUMMARY

 Method  Raw Detections  After Yaw Filter  After Deduplication Total Reduction %
M-DRAC             103                33                   11             89.3%
   SPF            2349              2306                  349             85.1%

✓ POST-PROCESSING COMPLETE

Processed files saved:
  - results/postprocessed_conflicts/mdrac_conflicts_processed.csv
  - results/postprocessed_conflicts/spf_conflicts_processed.csv
